[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/22_conv2d.ipynb)

# 🟠 Medium: 2D Convolution

Implement **2D convolution** from scratch.

### Signature
```python
def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    # x: (B, C_in, H, W), weight: (C_out, C_in, kH, kW)
    # Returns: (B, C_out, H_out, W_out)
```

### Rules
- Do NOT use `F.conv2d` or `nn.Conv2d`
- Support `stride` and `padding` parameters
- `F.pad` for zero-padding is allowed

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.3 MB/s eta 0:00:00


In [2]:

import torch
import torch.nn.functional as F

In [33]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    x = F.pad(x, (padding, padding, padding, padding))
    output = torch.zeros(x.shape[0], weight.shape[0], (x.shape[2] - weight.shape[2])//stride + 1, (x.shape[3] - weight.shape[3])//stride+ 1)
    for b in range(x.shape[0]):
      for k in range(weight.shape[0]):
        for i, r in enumerate(range(0, x.shape[2] - weight.shape[2] + 1, stride)):
          for j, c in enumerate(range(0, x.shape[3] - weight.shape[3] + 1, stride)):
            output[b,k,i,j] = torch.dot(weight[k].flatten(), x[b, :, r:r+weight.shape[2], c:c+weight.shape[3]].flatten())
        if bias is not None:
          output[b,k] += bias[k]
    return output
    # pass  # extract patches, apply kernel, handle stride/padding

In [39]:
def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    x = F.pad(x, (padding, padding, padding, padding))
    B, C_in, H, W = x.shape
    C_out, C_in, kH, kW = weight.shape
    output = torch.zeros(B, C_out, (H - kH)//stride + 1, (W - kW)//stride + 1)
    for b in range(B):
      for k in range(C_out):
        for i in range(output.shape[2]):
          for j in range(output.shape[3]):
            r, c = i*stride, j*stride
            patch = x[b, :, r:r+kH, c:c+kW]
            output[b,k,i,j] = torch.dot(weight[k].flatten(), patch.flatten())
        if bias is not None:
          output[b,k] += bias[k]
    return output



In [40]:
# 🧪 Debug
x = torch.randn(1, 3, 8, 8)
w = torch.randn(16, 3, 3, 3)
print('Output:', my_conv2d(x, w).shape)
print('Match:', torch.allclose(my_conv2d(x, w), F.conv2d(x, w), atol=1e-4))

Output: torch.Size([1, 16, 6, 6])
Match: True


In [41]:
# ✅ SUBMIT
from torch_judge import check
check('conv2d')


🧪 Testing: 2D Convolution (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (24.0ms)
  ✅ [2/5] Matches F.conv2d (12.4ms)
  ✅ [3/5] With padding (2.3ms)
  ✅ [4/5] With stride (1.7ms)
  ✅ [5/5] Gradient flow (1.9ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (42.3ms total)
  Progress saved. Run status() to see your dashboard.

